# Coversational Interface - Chatbot with LLM

Conversational interfaces such as chatbots and virtual assistants can be used to enhance the user experience for your customers. Chatbots uses natural language processing (NLP) and machine learning algorithms to understand and respond to user queries. Chatbots can be used in a variety of applications, such as customer service, sales, and e-commerce, to provide quick and efficient responses to users. They can be accessed through various channels such as websites, social media platforms, and messaging apps.

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip
!python3 -m pip3 install --upgrade pip

/usr/local/bin/python3: No module named pip3
CPU times: user 7.92 ms, sys: 13 ms, total: 20.9 ms
Wall time: 1.17 s


In [2]:
%pip install -U -q langchain langchain-core langchain_community langchain-ollama ipython-autotime --use-deprecated=legacy-resolver

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
time: 253 μs (started: 2025-12-09 13:24:59 -08:00)


In [3]:
# Restart kernel
from IPython.core.display import HTML

HTML("<script>Jupyter.notebook.kernel.restart()</script>")

time: 1.68 ms (started: 2025-12-09 13:24:59 -08:00)


## Initialize OLLAMA chatbot service
OLLAMA inference client is initialized to interact with the OLLAMA API for generating responses from the specified model.

In [4]:
my_model_ollama = "llama3.2"

from langchain_ollama.chat_models import ChatOllama

llm_chat = ChatOllama(
    model=my_model_ollama,
    base_url="http://localhost:11434",
    headers={"Content-Type": "application/json"},
    temperature=0.1,
)

llm_chat

ChatOllama(model='llama3.2', temperature=0.1, base_url='http://localhost:11434')

time: 2.27 s (started: 2025-12-09 13:24:59 -08:00)


Passing conversation state into and out a chain is vital when building a chatbot

In [5]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}


def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(llm_chat, get_session_history)

with_message_history

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x16cd179c0>, history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

time: 6.04 ms (started: 2025-12-09 13:25:01 -08:00)


In [6]:
from langchain_core.messages import HumanMessage

response = with_message_history.invoke(
    [HumanMessage(content="hi - i am krishna!")],
    config={"configurable": {"session_id": "1"}},
)

response

AIMessage(content="Namaste Krishna! It's lovely to meet you. Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2025-12-09T21:25:02.172752Z', 'done': True, 'done_reason': 'stop', 'total_duration': 662260458, 'load_duration': 101434458, 'prompt_eval_count': 33, 'prompt_eval_duration': 82146458, 'eval_count': 28, 'eval_duration': 468004500, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019b0500-ddb4-7010-bf97-1a6baa2f3e24-0', usage_metadata={'input_tokens': 33, 'output_tokens': 28, 'total_tokens': 61})

time: 687 ms (started: 2025-12-09 13:25:01 -08:00)


In [7]:
response.content

"Namaste Krishna! It's lovely to meet you. Is there something I can help you with, or would you like to chat?"

time: 2.17 ms (started: 2025-12-09 13:25:02 -08:00)


In [8]:
response = with_message_history.invoke(
    [HumanMessage(content="whats my name?")],
    config={"configurable": {"session_id": "1"}},
)
response

AIMessage(content='You told me earlier that your name is Krishna!', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2025-12-09T21:25:02.512811Z', 'done': True, 'done_reason': 'stop', 'total_duration': 313704708, 'load_duration': 67895875, 'prompt_eval_count': 75, 'prompt_eval_duration': 63927083, 'eval_count': 11, 'eval_duration': 177756334, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019b0500-e075-79f3-90f5-9860a68a7f3c-0', usage_metadata={'input_tokens': 75, 'output_tokens': 11, 'total_tokens': 86})

time: 318 ms (started: 2025-12-09 13:25:02 -08:00)


In [9]:
response.content

'You told me earlier that your name is Krishna!'

time: 819 μs (started: 2025-12-09 13:25:02 -08:00)


At this point the store has 1 key (session_id = '1') and its value is a list with 4 messages: [HumanMessage, AIMessage, HumanMessage, AIMessage]

In [10]:
print(store)

{'1': InMemoryChatMessageHistory(messages=[HumanMessage(content='hi - i am krishna!', additional_kwargs={}, response_metadata={}), AIMessage(content="Namaste Krishna! It's lovely to meet you. Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2025-12-09T21:25:02.172752Z', 'done': True, 'done_reason': 'stop', 'total_duration': 662260458, 'load_duration': 101434458, 'prompt_eval_count': 33, 'prompt_eval_duration': 82146458, 'eval_count': 28, 'eval_duration': 468004500, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019b0500-ddb4-7010-bf97-1a6baa2f3e24-0', usage_metadata={'input_tokens': 33, 'output_tokens': 28, 'total_tokens': 61}), HumanMessage(content='whats my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='You told me earlier that your name is Krishna!', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': 

In [11]:
store

{'1': InMemoryChatMessageHistory(messages=[HumanMessage(content='hi - i am krishna!', additional_kwargs={}, response_metadata={}), AIMessage(content="Namaste Krishna! It's lovely to meet you. Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2025-12-09T21:25:02.172752Z', 'done': True, 'done_reason': 'stop', 'total_duration': 662260458, 'load_duration': 101434458, 'prompt_eval_count': 33, 'prompt_eval_duration': 82146458, 'eval_count': 28, 'eval_duration': 468004500, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019b0500-ddb4-7010-bf97-1a6baa2f3e24-0', usage_metadata={'input_tokens': 33, 'output_tokens': 28, 'total_tokens': 61}), HumanMessage(content='whats my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='You told me earlier that your name is Krishna!', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': 

time: 1.04 ms (started: 2025-12-09 13:25:02 -08:00)


In [12]:
store["1"].messages[1]

AIMessage(content="Namaste Krishna! It's lovely to meet you. Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2025-12-09T21:25:02.172752Z', 'done': True, 'done_reason': 'stop', 'total_duration': 662260458, 'load_duration': 101434458, 'prompt_eval_count': 33, 'prompt_eval_duration': 82146458, 'eval_count': 28, 'eval_duration': 468004500, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019b0500-ddb4-7010-bf97-1a6baa2f3e24-0', usage_metadata={'input_tokens': 33, 'output_tokens': 28, 'total_tokens': 61})

time: 713 μs (started: 2025-12-09 13:25:02 -08:00)


## Create a Multi-Lingual Greeter Chatbot!

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You're an assistant who speaks in {language}. Translate the user input",
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

chain = prompt | llm_chat

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

lang = "French"
print(
    chain_with_history.invoke(
        {"language": lang, "question": "Hi my name is Krishna"},
        config={"configurable": {"session_id": "2"}},
    )
)

content="Bonjour, je m'appelle Krishna. Comment puis-je vous aider aujourd'hui ?" additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2025-12-09T21:25:03.032284Z', 'done': True, 'done_reason': 'stop', 'total_duration': 479949000, 'load_duration': 114710500, 'prompt_eval_count': 43, 'prompt_eval_duration': 60539167, 'eval_count': 18, 'eval_duration': 297846417, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'} id='lc_run--019b0500-e1d7-72c1-8cf7-267dd9258f10-0' usage_metadata={'input_tokens': 43, 'output_tokens': 18, 'total_tokens': 61}
time: 500 ms (started: 2025-12-09 13:25:02 -08:00)


In [14]:
print(
    chain_with_history.invoke(
        {"language": lang, "question": "What is my name?"},
        config={"configurable": {"session_id": "2"}},
    )
)

content='Votre nom est Krishna.' additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2025-12-09T21:25:03.256834Z', 'done': True, 'done_reason': 'stop', 'total_duration': 216404666, 'load_duration': 62065708, 'prompt_eval_count': 75, 'prompt_eval_duration': 64487500, 'eval_count': 6, 'eval_duration': 87310374, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'} id='lc_run--019b0500-e3bf-7421-841d-32e8dd197f43-0' usage_metadata={'input_tokens': 75, 'output_tokens': 6, 'total_tokens': 81}
time: 222 ms (started: 2025-12-09 13:25:03 -08:00)


In [15]:
store["2"].messages

[HumanMessage(content='Hi my name is Krishna', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Bonjour, je m'appelle Krishna. Comment puis-je vous aider aujourd'hui ?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2025-12-09T21:25:03.032284Z', 'done': True, 'done_reason': 'stop', 'total_duration': 479949000, 'load_duration': 114710500, 'prompt_eval_count': 43, 'prompt_eval_duration': 60539167, 'eval_count': 18, 'eval_duration': 297846417, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019b0500-e1d7-72c1-8cf7-267dd9258f10-0', usage_metadata={'input_tokens': 43, 'output_tokens': 18, 'total_tokens': 61}),
 HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Votre nom est Krishna.', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2025-12-09T21:25:03.256834Z', 'done': True, 'done_reason': 'stop', 'total_duration': 216404666

time: 5.2 ms (started: 2025-12-09 13:25:03 -08:00)
